## Long Short Term Memory (LSTM)

**Objective:** To predict Numerical financial stock values for next few days using LSTM Model (Long Short Term Memory). MSE Mean Square Error has been used here to evaluate  the model.

**Challenges:** There are around 6000 companies is being traded publicly. Its extremely difficult to predict which one is performing well, or will go up/down in next few days. Also, Its tedidious task to perform or analyse such tasks manually. Why can't we use AI, or the **basic** time series automated models for such tasks ?.

**Market Scope:** Approx 6000 companies
(US NASDAQ 2500, US NYSE 2200, India NSE 2500 Companies )


**Description:**
This script downloads stock data for multiple tickers, calculates technical indicators like RSI, MACD, and others, then uses an LSTM neural network to predict future stock prices. It also uses features like rolling mean and standard deviation, trains the model, and makes predictions for future prices.  

**The LSTM architecture** (Basic Model) in this script consists of three LSTM layers: the first layer has 100 units and returns sequences to pass to the next layer, followed by two LSTM layers with 50 units each. Dropout layers are added after each LSTM layer to reduce overfitting. Finally, two Dense layers are used for regression, with the last one outputting a single predicted value

In [5]:
!pip install pandas_ta
!pip install talib
!pip install yfinance
!pip install ace_tools

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.1/115.1 kB 2.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for pandas_ta: filename=pandas_ta-0.3.14b0-py3-none-any.whl size=218909 sha256=fc89ce026310d8d401193310e5456510923516f8017174b56563d32a0221894c
  Stored in directory: /root/.cache/pip/wheels/69/00/ac/f7fa862c34b0e2ef320175100c233377b4c558944f12474cf0
Successfully built pandas_ta
ERROR: Could not find a version that satisfies the requirement talib (from versions: none)
ERROR: No matching distribution found for talib


In [6]:
import pandas as pd
import os

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# Check the files in the specific directory
drive_path = '/content/drive/My Drive/Colab Notebooks'
print(os.listdir(drive_path))

sheet_name='US_Pull2'
file_path = '/content/drive/My Drive/Colab Notebooks/Stock_List.xlsx'
# Read the Excel sheet
ticker_data = pd.read_excel(file_path, sheet_name=sheet_name)

# Convert the 'ticker' column to a list
tickers = ticker_data['Ticker'].tolist()

# Now 'tickers' is a list of ticker symbols from your Excel sheet
number_of_tickers = len(tickers)
print("Number of tickers:", number_of_tickers)
#print(tickers)
print("Top 5 tickers:", tickers[:5])
print("Bottom 5 tickers:", tickers[-5:])
stock_desc = sheet_name

#end_date = '2024-10-18'

Mounted at /content/drive
['Current Portfolio_v4.ipynb', 'MACD stategy.ipynb', 'Backup', 'Stocks_v4.ipynb', 'MACD stategy_v1.ipynb', 'Test.ipynb', 'IBKR_API.ipynb', 'MACD stategy_v2.ipynb', 'Crypto_List_v1.ipynb', 'Crypto_v2.ipynb', 'MACD Cross Over_v1.ipynb', 'MACD Results_v1.xlsx', 'MACD Cross Over_v2.ipynb', 'MACD Results_v2.xlsx', 'Probabilistic_Model_v1.ipynb', 'Probabilistic_Model_v2.ipynb', 'MACD Results_v4_Main_US.xlsx', 'Current Portfolio_v5.ipynb', 'LSTM_Current Portfolio_v7.ipynb', 'MACD Results_v5_US1000_Active.xlsx', 'BB_Results_v5_US1000_8cons_Backtesting.xlsx', 'MACD Results_v5_US5000_MainStudy_v1_withResults.xlsx', 'BB_Results_v5_US1000_12cons_Live.xlsx', 'BB_Results_v5_US1000_12cons_Backtesting.xlsx', 'BB_Results_v5_US1000_Backtesting.xlsx', 'Probabilistic_Model_v3.ipynb', 'MACD Cross Over_v4.ipynb', 'MACD 162 Cross Over_v3.ipynb', 'VMAP_Three_Consecutive_v5_Active.xlsx', 'VMAP_6Three_Consecutive_v5_Active_v1.xlsx', 'Three_Consecutive_VMAP_v5.ipynb', 'Indicator_Score_v

In [7]:
from datetime import datetime, timedelta

# Get the current date and time
current_datetime = datetime.now()

# Calculate yesterday's date
yesterday = current_datetime - timedelta(days=0)

# Format yesterday's date as 'YYYY-MM-DD'
end_date = yesterday.strftime('%Y-%m-%d')

# Format current date and time as 'YYYY-MM-DD HH:MM:SS'
current_date_time = current_datetime.strftime('%Y-%m-%d %H:%M:%S')

# Print the results
print(f"Current Date and Time: {current_date_time}")
print(f"Yesterday's Date: {end_date}")


Current Date and Time: 2024-10-18 14:19:54
Yesterday's Date: 2024-10-18


***#First List - Time Series Prediction***


In [10]:

tickers = [
    "ADKO.VI", "AGR.VI", "AMAG.VI", "AMS.VI", "ANDR.VI", "ATH.VI", "ATS.VI", "AW2.VI",
    "BG.VI", "BHD.VI", "BKS.VI", "BKV.VI", "BTV.VI", "CAI.VI", "CLEN.VI", "DOC.VI",
    "EBS.VI", "EVN.VI", "FAA.VI", "FACC.VI", "FKA.VI", "FLU.VI", "FQT.VI", "GAGS.VI",
    "IIA.VI", "KTCG.VI", "LNZ.VI", "LTH.VI", "MAN.VI", "MARI.VI", "MMK.VI", "O2C.VI",
    "OBS.VI", "OBV.VI", "OMV.VI", "OTS.VI", "PAL.VI", "PMAG.VI", "POS.VI", "POST.VI",
    "PYT.VI", "RAT.VI", "RBI.VI", "RHIM.VI", "ROS.VI", "S300.VI", "SANT.VI", "SBO.VI",
    "SEM.VI", "SPI.VI", "STM.VI", "STR.VI", "SWUT.VI", "TKA.VI", "UBS.VI", "UIV.VI",
    "UQA.VI", "VER.VI", "VIG.VI", "VOE.VI", "VST.VI", "VVPS.VI", "WIE.VI", "WOL.VI",
    "WOLF.VI", "WPB.VI", "WXF.VI", "ZAG.VI"
]
#end_date = '2024-10-18'
stock_desc = "Vienna Stocks"

In [9]:
# # # List of tickers
tickers = ['RKLB', 'PHUN', 'TAOP', 'CXAI', 'YVRLF', 'DKNG', 'AAPL', 'PR', 'WDAY','CDW','LII', 'MEDP', 'MGM', 'ADI', 'JBL']
#end_date = '2024-10-18'
stock_desc = "MACD Portfolio"

In [15]:
# # # List of tickers
tickers = [
    'QUBT', 'RGTI', 'NTPC.NS', 'BHEL.NS', 'TMO', 'VLO', 'NOC', 'CXT', 'UTZ', 'SPCE', 'RTX',
    'TGT', 'STGR', 'ICCC', 'FRSH','NKE', 'SITE', 'JFU', 'RKLB', 'PHUN', 'TAOP', 'CXAI', 'YVRLF', 'DKNG', 'AAPL', 'PR', 'WDAY','CDW','LII', 'MEDP', 'MGM', 'ADI', 'JBL'
]
#end_date = '2024-10-18'
stock_desc = "Current and old Portfolio"

In [16]:
import yfinance as yf
import pandas_ta as ta
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.preprocessing import MinMaxScaler
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import mean_squared_error

import warnings
warnings.filterwarnings("ignore")

# Function to create the dataset
def create_dataset(data, time_step):
    X, Y = [], []
    for i in range(len(data) - time_step):
        X.append(data[i:(i + time_step), :-1])
        Y.append(data[i + time_step - 1, -1])
    return np.array(X), np.array(Y)

# Function to predict future prices
def predict_future_prices(model, last_data, steps, verbose=0):
    predictions = []
    current_data = last_data
    for _ in range(steps):
        pred = model.predict(current_data[np.newaxis, :, :-1], verbose=verbose)
        predictions.append(pred[0, 0])
        current_data = np.append(current_data[1:], np.concatenate((current_data[-1:, :-1], pred), axis=1), axis=0)
    return np.array(predictions)
# #
# # # List of tickers
# tickers = [
#     'QUBT', 'RGTI', 'NTPC.NS', 'BHEL.NS', 'TMO', 'VLO', 'NOC', 'CXT', 'UTZ', 'SPCE', 'RTX',
#     'TGT', 'STGR', 'ICCC', 'FRSH','NKE', 'SITE', 'JFU', 'RKLB', 'PHUN', 'TAOP', 'CXAI', 'YVRLF'
# ]

# tickers = [
#     'NTPC.NS', 'BHEL.NS'
# ]

results = []#
filtered_df = data = []
i = 0

for ticker in tickers:
    try:
        # Download stock data
        #data = yf.download(ticker, period='2y')
        import datetime
        # Set the end date and the start date

        start_date = (datetime.datetime.strptime(end_date, '%Y-%m-%d') - datetime.timedelta(days=2 * 365)).strftime(
            '%Y-%m-%d')
        # Download stock data
        data = []
        data = yf.download(ticker, start=start_date, end=end_date)

        i = i + 1
        print(f"Processed {ticker}: {i} out of {len(tickers)} tickers")

        # Calculate technical indicators
        data['RSI'] = ta.rsi(data['Close'])
        macd = ta.macd(data['Close'])
        data['MACD'] = macd['MACD_12_26_9']
        data['MACD_signal'] = macd['MACDs_12_26_9']
        data['MACD_hist'] = macd['MACDh_12_26_9']
        data['EMA'] = ta.ema(data['Close'], length=5)
        data['OBV'] = ta.obv(data['Close'], data['Volume'])
        bb = ta.bbands(data['Close'])
        data['BB_upper'] = bb['BBU_5_2.0']
        data['BB_middle'] = bb['BBM_5_2.0']
        data['BB_lower'] = bb['BBL_5_2.0']
        data['Stochastic'] = ta.stoch(data['High'], data['Low'], data['Close'])['STOCHk_14_3_3']
        data['ATR'] = ta.atr(data['High'], data['Low'], data['Close'])
        data['ADL'] = ta.ad(data['High'], data['Low'], data['Close'], data['Volume'])
        data['CCI'] = ta.cci(data['High'], data['Low'], data['Close'], length=20)
        data['Williams_%R'] = ta.willr(data['High'], data['Low'], data['Close'])

        # Add additional features
        data['Roll_Mean_5'] = data['Close'].rolling(window=5).mean()
        data['Roll_Std_5'] = data['Close'].rolling(window=5).std()
        data['RSI_Roll_Change_5'] = data['RSI'] - data['RSI'].shift(5)
        data['MACD_hist_Roll_Change_5'] = data['MACD_hist'] - data['MACD_hist'].shift(5)
        data['OBV_Roll_Change_5'] = data['OBV'] - data['OBV'].shift(5)
        data['Price_Roll_Change_5'] = data['Close'] - data['Close'].shift(5)
        data['Price_Diff'] = data['High'] - data['Low']

        #New Features Added:
        # data['RSI_Chg'] = ((data['RSI'] - data['RSI'].shift(5)) / data['RSI'].shift(5)) * 100
        # data['RSI_Chg3'] = ((data['RSI'] - data['RSI'].shift(3)) / data['RSI'].shift(3)) * 100
        # data['MACD_hist_Chg'] = ((data['MACD_hist'] - data['MACD_hist'].shift(5)) / data['MACD_hist'].shift(5)) * 100
        # data['OBV_Chg'] = ((data['OBV'] - data['OBV'].shift(5)) / data['OBV'].shift(5)) * 100
        # data['Price_Chg'] = ((data['Close'] - data['Close'].shift(5)) / data['Close'].shift(5)) * 100
        # #data['VMAP_Chg'] = ((data['VWAP'] - data['VWAP'].shift(5)) / data['VWAP'].shift(5)) * 100

        # Forward-fill any NaN values in indicators like Klinger Oscillator, Volume RSI, RSI
        # #data['Klinger Oscillator'] = data['Klinger Oscillator'].ffill()
        # data['Volume RSI'] = data['Volume RSI'].ffill()
        # data['RSI'] = data['RSI'].ffill()
        # data['BB_%target'] = ((data['BB_upper'] - data['Close']) / data['BB_upper']) * 100

        # Remove the 'Open', 'High', 'Low', and 'Adj Close' columns
        data = data.drop(columns=['High', 'Low', 'Adj Close'])
        data.dropna(inplace=True)

        # Prepare the data
        #features = ['Open', 'RSI', 'MACD', 'MACD_signal', 'MACD_hist', 'EMA', 'OBV']
        features = ['Open', 'RSI', 'MACD', 'MACD_signal', 'MACD_hist', 'EMA', 'OBV', 'Roll_Mean_5', 'Roll_Std_5', 'RSI_Roll_Change_5',
                    'MACD_hist_Roll_Change_5', 'OBV_Roll_Change_5', 'Price_Roll_Change_5', 'Price_Diff']
        target = 'Close'
        scaler = MinMaxScaler()
        scaled_features = scaler.fit_transform(data[features])
        scaled_target = scaler.fit_transform(data[[target]])
        scaled_data = np.hstack((scaled_features, scaled_target))
        train_data = scaled_data[:-60]
        test_data = scaled_data[-60:]
        time_step = 5
        X_train, y_train = create_dataset(train_data, time_step)
        X_test, y_test = create_dataset(test_data, time_step)

        # Build the LSTM model
        model = Sequential()
        model.add(LSTM(100, return_sequences=True, input_shape=(time_step, len(features))))
        model.add(Dropout(0.2))
        model.add(LSTM(50, return_sequences=True))
        model.add(Dropout(0.2))
        model.add(LSTM(50, return_sequences=False))
        model.add(Dropout(0.2))
        model.add(Dense(50))
        model.add(Dense(1))

        model.compile(optimizer='adam', loss='mean_squared_error')
        model.fit(X_train, y_train, batch_size=32, epochs=30, validation_split=0.1, verbose=0)

        # Make predictions
        train_predict = model.predict(X_train)
        test_predict = model.predict(X_test)
        train_predict = scaler.inverse_transform(np.concatenate((np.zeros((train_predict.shape[0], len(features))), train_predict), axis=1))[:, -1]
        test_predict = scaler.inverse_transform(np.concatenate((np.zeros((test_predict.shape[0], len(features))), test_predict), axis=1))[:, -1]
        train_mse = mean_squared_error(scaler.inverse_transform(scaled_target[:-60][time_step:])[:, 0], train_predict)
        test_mse = mean_squared_error(scaler.inverse_transform(scaled_target[-60:])[:, 0][time_step:], test_predict)

        # Predict future prices
        last_data = scaled_data[-time_step:]
        future_predictions = predict_future_prices(model, last_data, 6, verbose=0)
        future_predictions_actual = scaler.inverse_transform(np.concatenate((np.zeros((future_predictions.shape[0], len(features))), future_predictions.reshape(-1, 1)), axis=1))[:, -1]
        future_dates = pd.date_range(start=data.index[-1], periods=6, freq='B')[0:]
        future_df = pd.DataFrame({'Date': future_dates, 'Predicted Closing Price': future_predictions_actual})
        future_df['Predicted % Change'] = future_df['Predicted Closing Price'].pct_change() * 100

        latest_rsi = data['RSI'].iloc[-1]
        MACD_chg = data['MACD_hist_Roll_Change_5'].iloc[-1]


        results.append({
            'ticker': ticker,
            'train_mse': train_mse,
            'test_mse': test_mse,
            'future_predictions': future_df,
            'latest_date': data.index.max().strftime('%Y-%m-%d'),  # Include latest date here
            'latest_rsi': latest_rsi,  # Include latest RSI value here
            'MACD_chg': MACD_chg

        })

    except Exception as e:
        print(f"Error processing {ticker}: {e}")
        continue

    try:

        # Combine results into a single DataFrame
        final_columns = ['Stock Name', 'Train MSE', 'Test MSE'] + \
                        [f'{date.date()}' for date in future_dates] + \
                        [f'{date.date()} % Change' for date in future_dates[1:]]
        final_df = pd.DataFrame(columns=final_columns)

        for result in results:
            row = {
                'Stock Name': result['ticker'],
                'Train MSE': result['train_mse'],
                'Test MSE': result['test_mse'],
                'Date': result['latest_date'],
                'rsi': result['latest_rsi'],
                'MACD_chg': result['MACD_chg']
            }
            for date in future_dates:
                row[f'{date.date()}'] = result['future_predictions'].loc[result['future_predictions']['Date'] == date, 'Predicted Closing Price'].values[0]
            for date in future_dates[1:]:
                row[f'{date.date()} % Change'] = result['future_predictions'].loc[result['future_predictions']['Date'] == date, 'Predicted % Change'].values[0]

            first_predicted_price = row[f'{future_dates[0].date()}']
            row['Norm_Test_MSE'] = row['Test MSE'] / first_predicted_price
            row['Overall %'] = sum(row[f'{date.date()} % Change'] for date in future_dates[1:4])
            row['+ve Count'] = sum(1 for date in future_dates[1:4] if row[f'{date.date()} % Change'] > 0)
            row['-ve Count'] = sum(1 for date in future_dates[1:4] if row[f'{date.date()} % Change'] < 0)

            final_df = pd.concat([final_df, pd.DataFrame([row])], ignore_index=True)

        final_df.drop(final_df.columns[4:9], axis=1, inplace=True)
        # Sort the DataFrame by 'Norm_Test_MSE'
        final_df = final_df.sort_values(by='Norm_Test_MSE')

        # Display the final DataFrame
        #display(final_df)

        final_df = final_df[(final_df['Norm_Test_MSE'] < 0.30) &
                       ((final_df['Overall %'] > 2) | (final_df['Overall %'] < -2))]

        # Display the filtered DataFrame
        #display(final_df)

        #from google.colab import drive
        #drive.mount('/content/drive')

        # Save the DataFrame to an Excel file
        #final_df.to_excel(file_path, index=False)

        #print(f"DataFrame saved to {file_path}")
        #32 mins, 106 stocks.

    except Exception as e:
        #print(f"Error processing {ticker}: {e}")
        continue
display(final_df)

[*********************100%***********************]  1 of 1 completed


Processed QUBT: 1 out of 33 tickers
13/13 ━━━━━━━━━━━━━━━━━━━━ 1s 29ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step 


[*********************100%***********************]  1 of 1 completed


Processed RGTI: 2 out of 33 tickers
13/13 ━━━━━━━━━━━━━━━━━━━━ 1s 29ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step 


[*********************100%***********************]  1 of 1 completed


...

[*********************100%***********************]  1 of 1 completed


Processed ADI: 32 out of 33 tickers
13/13 ━━━━━━━━━━━━━━━━━━━━ 1s 29ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step 


[*********************100%***********************]  1 of 1 completed


Processed JBL: 33 out of 33 tickers
13/13 ━━━━━━━━━━━━━━━━━━━━ 1s 30ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step 


,Stock Name,Train MSE,Test MSE,2024-10-17,2024-10-18,2024-10-21,2024-10-22,2024-10-23,2024-10-24,2024-10-18 % Change,...,2024-10-22 % Change,2024-10-23 % Change,2024-10-24 % Change,Date,rsi,MACD_chg,Norm_Test_MSE,Overall %,+ve Count,-ve Count
0,QUBT,0.008851,0.002989,0.766851,0.822251,0.871705,0.925059,0.953064,0.953064,7.224299,...,6.120636,3.027393,0.0,2024-10-17,73.179905,0.014265,0.003897,19.359402,3.0,0.0
1,RGTI,0.024097,0.005494,0.970166,1.034454,1.082033,1.100473,1.105750,1.105750,6.626479,...,1.704250,0.479533,0.0,2024-10-17,65.308046,0.018652,0.005663,12.930134,3.0,0.0
2,NTPC.NS,60.649898,68.646349,431.330136,429.385897,425.873079,422.934115,421.432274,421.432274,-0.450754,...,-0.690103,-0.355100,0.0,2024-10-17,47.138256,0.185656,0.159150,-1.958960,0.0,3.0
3,BHEL.NS,101.879667,144.740313,280.861247,277.555292,273.278042,269.899724,266.706074,266.706074,-1.177078,...,-1.236220,-1.183273,0.0,2024-10-17,37.167781,-0.400252,0.515345,-3.954342,0.0,3.0
4,TMO,135.427355,155.014942,591.563666,591.799615,591.301286,591.750855,594.494379,594.494379,0.039886,...,0.076031,0.463628,0.0,2024-10-17,46.901812,1.480772,0.262043,0.031710,2.0,1.0
5,VLO,17.537390,17.984033,136.638329,135.279566,134.148918,134.034525,134.280871,134.280871,-0.994424,...,-0.085273,0.183793,0.0,2024-10-17,46.866312,-1.086293,0.131618,-1.915482,0.0,3.0
6,NOC,72.097671,41.125449,527.999797,528.433356,528.021033,527.608314,527.577844,527.577844,0.082113,...,-0.078163,-0.005775,0.0,2024-10-17,51.269994,-0.235662,0.077889,-0.074077,1.0,2.0
7,CXT,2.230077,3.637006,55.289316,55.633509,56.192642,56.570762,56.893019,56.893019,0.622532,...,0.672899,0.569652,0.0,2024-10-17,58.677545,0.522816,0.065781,2.300460,3.0,0.0
8,UTZ,0.211102,0.323788,17.464780,17.621382,17.697153,17.770996,17.824576,17.824576,0.896671,...,0.417259,0.301499,0.0,2024-10-17,54.348921,0.150185,0.018539,1.743929,3.0,0.0
9,SPCE,38.993694,33.524355,14.992521,15.826422,16.145253,16.206270,16.122923,16.122923,5.562115,...,0.377925,-0.514287,0.0,2024-10-17,62.675179,0.110921,2.236072,7.954588,3.0,0.0


In [19]:
import yfinance as yf
import pandas as pd

# Define the stock ticker
ticker = "SPCE"

# Download the stock data for the last 5 days
data = yf.download(ticker, period='5d')

# Display the results
print("Last 5 Days of Daily Stock Data for EBS.VI:")
print(data)

[*********************100%***********************]  1 of 1 completed

Last 5 Days of Daily Stock Data for EBS.VI:
            Open   High    Low  Close  Adj Close   Volume
Date                                                     
2024-10-14  6.28  6.590  6.133  6.590      6.590  1517400
2024-10-15  6.58  6.865  6.540  6.770      6.770  1174200
2024-10-16  6.89  7.400  6.860  7.150      7.150  1649700
2024-10-17  7.15  7.170  6.840  7.060      7.060   958400
2024-10-18  7.12  7.606  7.090  7.325      7.325   834821
